# Advanced Patterns

## What you'll learn

- Build diamond DAGs that diverge and reconverge
- Control work package sizes with `artifacts_per_unit` (batching)
- Implement iterative refinement loops with standard Python control flow

**Prerequisites:** [Sources and Chains](01-sources-and-chains.ipynb), [Branching and Merging](02-branching-and-merging.ipynb), [Metrics and Filtering](03-metrics-and-filtering.ipynb)
**Estimated time:** 15 minutes

---

The previous tutorials covered individual building blocks: sources, chains,
fan-out, merging, filtering, and multi-input operations. This tutorial
combines those primitives into composite patterns you'll use in real
pipelines.

| Pattern | Combines | Description |
|---------|----------|-------------|
| **Diamond DAG** | Fan-out + Merge + Chain | Diverge into parallel branches, reconverge, then continue |
| **Batching** | Any pattern + execution config | Group multiple artifacts into fewer execution units |
| **Iterative refinement** | Generation + Scoring + Filtering + Transformation in a loop | Repeated generate → score → filter → modify cycles |

In [ ]:
from __future__ import annotations

from artisan.operations.curator import Filter, Merge
from artisan.operations.examples import (
    DataGenerator,
    DataTransformer,
    MetricCalculator,
)
from artisan.orchestration import PipelineManager
from artisan.utils import tutorial_setup
from artisan.utils.logging import configure_logging
from artisan.visualization import (
    build_macro_graph,
    build_micro_graph,
    inspect_metrics,
    inspect_pipeline,
)

configure_logging()

> **Graph legend:** See [Sources and Chains](01-sources-and-chains.ipynb) for box/arrow key.

## Diamond DAG

A diamond is the simplest composite pattern: one source fans out to two
independent branches, both branches process the same inputs differently,
then a Merge step reconverges them before downstream processing continues.

```
              ┌─── transform (seed=100) ───┐
generate ─────┤                             ├─── merge ─── metrics
              └─── transform (seed=200) ───┘
```

**When to use:** You need to apply two independent transformations to the
same inputs and then process the combined results — for example, generating
candidates via two strategies and then scoring them together.

In [ ]:
env_diamond = tutorial_setup("diamond_dag")

In [ ]:
pipeline = PipelineManager.create(
    name="diamond_dag",
    delta_root=env_diamond.delta_root,
    staging_root=env_diamond.staging_root,
    working_root=env_diamond.working_root,
)
output = pipeline.output

# Source: generate 2 datasets
pipeline.run(operation=DataGenerator, name="generate", params={"count": 2, "seed": 42})

# Branch A: transform with one strategy
pipeline.run(
    operation=DataTransformer,
    name="transform_a",
    inputs={"dataset": output("generate", "datasets")},
    params={"seed": 100, "output_prefix": "branch_a_"},
)

# Branch B: transform with a different strategy
pipeline.run(
    operation=DataTransformer,
    name="transform_b",
    inputs={"dataset": output("generate", "datasets")},
    params={"seed": 200, "output_prefix": "branch_b_"},
)

# Reconverge: merge both branches into a single stream
pipeline.run(
    operation=Merge,
    name="merge",
    inputs={
        "branch_a": output("transform_a", "dataset"),
        "branch_b": output("transform_b", "dataset"),
    },
)

# Continue: score the combined results
pipeline.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("merge", "merged")},
)

result_diamond = pipeline.finalize()

In [ ]:
build_macro_graph(env_diamond.delta_root)

The macro graph shows the diamond shape: `data_generator` feeds both
`data_transformer` steps (fan-out), which converge on `merge`, and then
`metric_calculator` processes the unified stream. The Merge step produces no
new artifacts — it unions the two branches using passthrough semantics, so
`metric_calculator` receives all 4 datasets (2 from each branch).

In [ ]:
inspect_pipeline(env_diamond.delta_root)

The pipeline overview confirms the flow: step 0 produced 2 datasets,
steps 1 and 2 each transformed those 2 into 2 more, Merge passed all 4
through, and `metric_calculator` scored each one.

In [ ]:
build_micro_graph(env_diamond.delta_root)

## Batching

By default, each artifact gets its own execution unit. When per-unit startup
cost is high (container launch, GPU initialization, loading a large model),
this creates unnecessary overhead. The `artifacts_per_unit` parameter groups
multiple input artifacts into a single execution unit.

**When to use:** Processing many small artifacts where the fixed cost per
execution dominates the per-artifact cost.

| `artifacts_per_unit` | Behavior |
|---------------------|----------|
| 1 (default) | One execution unit per artifact |
| N | Groups of N artifacts share an execution unit |
| N > total | All artifacts in one execution unit |

In [ ]:
env_batch = tutorial_setup("batching")

In [ ]:
pipeline = PipelineManager.create(
    name="batching",
    delta_root=env_batch.delta_root,
    staging_root=env_batch.staging_root,
    working_root=env_batch.working_root,
)
output = pipeline.output

# Generate 6 datasets
pipeline.run(operation=DataGenerator, name="generate", params={"count": 6, "seed": 42})

# Transform with batching: 3 artifacts per execution unit → 2 units total
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    params={"seed": 100},
    execution={"artifacts_per_unit": 3},
)

result_batch = pipeline.finalize()

Without batching, step 1 would have dispatched 6 execution units (one per
dataset). With `artifacts_per_unit=3`, it dispatched 2 units instead — each
processing 3 datasets. Check the log output above: step 1 reports
"6 artifacts → 2 execution units."

In [ ]:
inspect_pipeline(env_batch.delta_root)

In [ ]:
build_macro_graph(env_batch.delta_root)

The macro graph looks the same whether batching is used or not — batching
is an execution concern, not a DAG structure concern. The provenance stepper
below reveals the difference: step 1 has fewer execution records (grey
boxes) than output artifacts.

In [ ]:
build_micro_graph(env_batch.delta_root)

## Iterative refinement

This is the most powerful composite pattern. Many real workflows follow
a loop: generate candidates, score them, keep the best, modify survivors,
and repeat. Artisan handles this with a standard Python `for` loop around
`pipeline.run()` calls — no special API needed.

```
Round 0:  generate ──→ score ──→ filter ──→ transform ──┐
Round 1:              score ←──────────────────────────────┘
                        ↓
                      filter ──→ transform ──┐
Round 2:              score ←─────────────────┘
                        ↓
                      filter ──→ transform (final)
```

Each iteration adds new steps to the same pipeline. Provenance tracks
lineage across all rounds automatically.

**When to use:** Design optimization, evolutionary search, iterative
screening — any workflow where you repeatedly score, select, and regenerate.

### Building one round

Before looping, let's build a single round to understand each piece. The
four steps are:

1. **Generate** — create initial candidate datasets
2. **Score** — compute metrics on each candidate
3. **Filter** — keep candidates that meet a threshold
4. **Transform** — modify survivors for the next round

In [ ]:
env_single = tutorial_setup("cycling_single_round")

In [ ]:
pipeline = PipelineManager.create(
    name="cycling_single_round",
    delta_root=env_single.delta_root,
    staging_root=env_single.staging_root,
    working_root=env_single.working_root,
)
output = pipeline.output

# 1. Generate 6 initial candidates
pipeline.run(operation=DataGenerator, name="generate", params={"count": 6, "seed": 42})
datasets = output("generate", "datasets")

# 2. Score each candidate
scored = pipeline.run(
    operation=MetricCalculator, name="score", inputs={"dataset": datasets}
)

# 3. Filter: keep candidates with max value >= 0
filtered = pipeline.run(
    operation=Filter,
    name="filter",
    inputs={
        "passthrough": datasets,
        "scores": output("score", "metrics"),
    },
    params={
        "criteria": [
            {"metric": "scores.distribution.max", "operator": "ge", "value": 0},
        ]
    },
)

# 4. Transform survivors
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("filter", "passthrough")},
    params={"seed": 100},
)

result_single = pipeline.finalize()

In [ ]:
inspect_metrics(env_single.delta_root)

The metrics table shows each candidate's scores. The Filter step used
`scores.distribution.max >= 0` to decide which candidates survive — the
`"scores"` prefix tells Filter to look in the explicit metric input role.
Candidates that pass continue to the Transform step; those that fail are
dropped.

In [ ]:
build_macro_graph(env_single.delta_root)

In [ ]:
build_micro_graph(env_single.delta_root)

The graph shows the single-round flow: generate → score → filter →
transform. Notice that Filter receives both the datasets (as passthrough)
and the metrics (as an explicit `"scores"` role), using lineage matching
to pair each dataset with its corresponding metric.

### Looping over multiple rounds

Now wrap the score → filter → transform sequence in a `for` loop. Each
round's output feeds into the next round's input. The key insight:
`pipeline.run()` returns a `StepResult` with `.output()` references that
can be passed directly to the next call. A Python variable reassignment
(`datasets = ...`) is all you need to chain rounds together.

In [ ]:
env_cycle = tutorial_setup("cycling_multi_round")

N_ROUNDS = 3

In [ ]:
pipeline = PipelineManager.create(
    name="cycling_multi_round",
    delta_root=env_cycle.delta_root,
    staging_root=env_cycle.staging_root,
    working_root=env_cycle.working_root,
)
output = pipeline.output

# Initial generation
pipeline.run(operation=DataGenerator, name="generate", params={"count": 6, "seed": 42})
datasets = output("generate", "datasets")

# Iterative refinement: score → filter → transform, repeated N rounds
for round_num in range(N_ROUNDS):
    # Score current candidates
    scored_name = f"score_r{round_num}"
    pipeline.run(
        operation=MetricCalculator, name=scored_name, inputs={"dataset": datasets}
    )

    # Filter: keep candidates above threshold
    filter_name = f"filter_r{round_num}"
    pipeline.run(
        operation=Filter,
        name=filter_name,
        inputs={
            "passthrough": datasets,
            "scores": output(scored_name, "metrics"),
        },
        params={
            "criteria": [
                {"metric": "scores.distribution.max", "operator": "ge", "value": 0},
            ]
        },
    )

    # Transform survivors for the next round
    transform_name = f"transform_r{round_num}"
    pipeline.run(
        operation=DataTransformer,
        name=transform_name,
        inputs={"dataset": output(filter_name, "passthrough")},
        params={"seed": (round_num + 1) * 100},
    )

    # Feed modified datasets into the next round
    datasets = output(transform_name, "dataset")

result_cycle = pipeline.finalize()

The loop ran 3 rounds, adding 3 steps per round (score, filter, transform)
for 10 total steps (1 initial generation + 3 × 3). Each round's Transform
output became the next round's Score input through the `datasets` variable
reassignment.

No special framework API was needed — the loop is plain Python. The
framework handles provenance tracking across rounds automatically.

In [ ]:
inspect_pipeline(env_cycle.delta_root)

The pipeline overview shows all 10 steps. Each group of three
(metric_calculator → filter → data_transformer) is one round. Watch the
artifact counts: if the filter drops candidates in early rounds, later
rounds process fewer artifacts.

In [ ]:
inspect_metrics(env_cycle.delta_root)

The metrics table shows scores from every round. Metrics from round 0 are
computed on the initial candidates; round 1 metrics come from the
transformed survivors, and so on. Comparing across rounds reveals whether
the iterative process is improving candidates.

In [ ]:
build_macro_graph(env_cycle.delta_root)

The macro graph shows the repeating pattern across rounds. Each round's
score → filter → transform subgraph is connected to the previous round's
output, forming a chain of refinement cycles. Provenance arrows trace
lineage from the initial candidates through every transformation.

In [ ]:
build_micro_graph(env_cycle.delta_root)

Use the stepper to walk through rounds one step at a time. This is
the best way to verify that lineage is tracked correctly across
round boundaries — each round's Transform outputs should trace back
through the Filter to the previous round's Transform outputs.

## Summary

This tutorial covered three composite patterns that build on the
fundamentals from previous tutorials:

- **Diamond DAG** — fan-out into parallel branches, merge, and continue.
  Combines [fan-out](02-branching-and-merging.ipynb) and
  [merge](02-branching-and-merging.ipynb) with downstream processing.
- **Batching** — `artifacts_per_unit` controls how many artifacts share
  an execution unit. Reduces overhead without changing the DAG.
- **Iterative refinement** — a Python `for` loop around
  `pipeline.run()` calls implements score → filter → transform cycles.
  No special API needed; provenance tracks lineage across all rounds
  automatically.

**Key takeaway:** Artisan's composability comes from Python, not from
framework abstractions. `OutputReference` objects are the wiring
mechanism, and standard control flow (loops, conditionals) is how you
build complex topologies.

**Next:** [Error Visibility](06-error-visibility.ipynb) — handling empty filters, inspecting skipped steps, and soft gating